<a href="https://colab.research.google.com/github/jishnujs1990/Study/blob/master/Sentimental_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Objective:
Develop machine learning models to classify emotions in text samples.

#Key components to be fulfilled :**
##1. Loading and Preprocessing (3 marks)**

● Load the dataset and perform necessary preprocessing steps. This should include text
cleaning, tokenization, and removal of stopwords. Explain the preprocessing techniques
used and their impact on model performance.

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# 1. Load Dataset
df = pd.read_csv('nlp_dataset.csv')

print("--- Dataset Overview ---")
print(df.head())
print("\nDataset Shape:", df.shape)
print("\nClass Distribution:\n", df['Emotion'].value_counts())

# Initialize Lemmatizer and Stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # 1. Convert to lowercase
    text = text.lower()
    # 2. Remove punctuation and non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Tokenize and remove stopwords + lemmatize
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(cleaned_words)

# Apply Preprocessing
df['Cleaned_Comment'] = df['Comment'].apply(preprocess_text)

print("\n--- Sample Preprocessed Data ---")
print(df[['Comment', 'Cleaned_Comment', 'Emotion']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


--- Dataset Overview ---
                                             Comment Emotion
0  i seriously hate one subject to death but now ...    fear
1                 im so full of life i feel appalled   anger
2  i sit here to write i start to dig out my feel...    fear
3  ive been really angry with r and i feel like a...     joy
4  i feel suspicious if there is no one outside l...    fear

Dataset Shape: (5937, 2)

Class Distribution:
 Emotion
anger    2000
joy      2000
fear     1937
Name: count, dtype: int64

--- Sample Preprocessed Data ---
                                             Comment  \
0  i seriously hate one subject to death but now ...   
1                 im so full of life i feel appalled   
2  i sit here to write i start to dig out my feel...   
3  ive been really angry with r and i feel like a...   
4  i feel suspicious if there is no one outside l...   

                                     Cleaned_Comment Emotion  
0  seriously hate one subject death feel reluctan..

##2. Feature Extraction (2 marks):
● Implement feature extraction using CountVectorizer or TfidfVectorizer. Describe how the
chosen method transforms the text data into numerical features.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Split Features and Target
X = df['Cleaned_Comment']
y = df['Emotion']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit and Transform on Train data, Transform Test data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF Train Matrix Shape: {X_train_tfidf.shape}")
print(f"TF-IDF Test Matrix Shape:  {X_test_tfidf.shape}")

TF-IDF Train Matrix Shape: (4749, 5000)
TF-IDF Test Matrix Shape:  (1188, 5000)


##3. Model Development (2 marks):
● Train the following machine learning models
a)Naive Bayesb)Support Vector Machine

In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

# a) Naive Bayes Model
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

# b) Support Vector Machine Model
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_preds = svm_model.predict(X_test_tfidf)

print("Both models trained successfully.")

Both models trained successfully.


##4. Model Comparison (2 marks)
● Evaluate the model using appropriate metrics (e.g., accuracy, F1-score). Provide a brief
explanation of the chosen model and its suitability for emotion classification.

In [5]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    return {'Model': name, 'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}

# Calculate Metrics
nb_metrics = evaluate_model("Multinomial Naive Bayes", y_test, nb_preds)
svm_metrics = evaluate_model("Support Vector Machine (Linear)", y_test, svm_preds)

comparison_df = pd.DataFrame([nb_metrics, svm_metrics])
print("=== Model Comparison Metrics ===")
print(comparison_df.to_string(index=False))

print("\n=== Detailed Classification Report: Naive Bayes ===")
print(classification_report(y_test, nb_preds))

print("\n=== Detailed Classification Report: SVM ===")
print(classification_report(y_test, svm_preds))

=== Model Comparison Metrics ===
                          Model  Accuracy  Precision   Recall  F1-Score
        Multinomial Naive Bayes  0.909933   0.910275 0.909933  0.909978
Support Vector Machine (Linear)  0.931818   0.931818 0.931818  0.931770

=== Detailed Classification Report: Naive Bayes ===
              precision    recall  f1-score   support

       anger       0.89      0.91      0.90       400
        fear       0.91      0.92      0.91       388
         joy       0.93      0.90      0.91       400

    accuracy                           0.91      1188
   macro avg       0.91      0.91      0.91      1188
weighted avg       0.91      0.91      0.91      1188


=== Detailed Classification Report: SVM ===
              precision    recall  f1-score   support

       anger       0.93      0.92      0.92       400
        fear       0.94      0.93      0.93       388
         joy       0.93      0.95      0.94       400

    accuracy                           0.93      1188


##Support Vector Machine (Linear) outperforms Multinomial Naive Bayes across all key metrics